# BUSINESS UNDERSTANDING

#### > Spotify adalah platform streaming musik yang menyediakan jutaan lagu dari berbagai artis, album, dan genre.

#### > Popularity adalah skor yang menunjukkan seberapa populer sebuah lagu di Spotify. Nilainya berada di rentang 0 sampai 100.

#### > Dalam konteks bisnis, platform musik perlu memahami karakteristik lagu populer agar bisa membuat rekomendasi playlist yang lebih relevan untuk pengguna.

#### > Project ini bertujuan mencari pola dari audio features, genre, dan popularity score untuk menyusun insight serta rekomendasi playlist strategy.

Dataset Kaggle: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset

### Business Problem

Bagaimana karakteristik audio dan genre lagu memengaruhi popularitas, dan bagaimana insight tersebut dapat digunakan untuk membuat strategi playlist yang lebih relevan?

### Pertanyaan Bisnis

1. Bagaimana distribusi popularity lagu Spotify?
2. Genre apa yang memiliki rata-rata popularity tertinggi?
3. Apakah lagu explicit lebih populer dibanding non-explicit?
4. Audio features apa yang membedakan lagu populer dan tidak populer?
5. Fitur apa yang paling berpengaruh dalam memprediksi lagu populer?

# Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import os

csv_files = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.lower().endswith('.csv'):
            csv_files.append(os.path.join(dirname, filename))

print("CSV files ditemukan:")
for path in csv_files:
    print(path)

if len(csv_files) == 0:
    raise FileNotFoundError("Tidak ada file CSV di /kaggle/input. Pastikan dataset Spotify dari Kaggle sudah ditambahkan.")

data_path = csv_files[0]
df = pd.read_csv(data_path)
df.head(10)

# EDA (Exploratory Data Analysis)

In [ ]:
# ukuran data (baris, kolom)
df.shape

In [ ]:
# ini lebih jelas untuk melihat jumlah baris, kolom, dan tipe data
df.info()

### Penjelasan Kolom

> `popularity` adalah skor popularitas lagu dari 0 sampai 100.

> `danceability` menunjukkan seberapa cocok lagu untuk menari.

> `energy` menunjukkan tingkat intensitas atau energi lagu.

> `valence` menunjukkan mood lagu. Nilai tinggi berarti mood lebih positif atau ceria.

> `acousticness` menunjukkan seberapa akustik karakter lagu.

> `instrumentalness` menunjukkan kemungkinan lagu bersifat instrumental.

> `liveness` menunjukkan kemungkinan lagu direkam secara live.

> `tempo` menunjukkan kecepatan lagu dalam BPM.

> `key` adalah Nada dasar lagu, direpresentasikan dengan angka.
contoh  jika key = 9, berarti lagu berada di nada dasar A.

> `duration_ms` adalah lama waktu lagu dalam milisecond

> `loudnes` adalah Tingkat keras atau pelan suara lagu, diukur dalam decibel / dB, semakin mendekati 0 itu cukup keras

> `Mode` adalah jenis tanggal nada lagu ada major dan minor, major biasanya lagu positif, minor biasanya lagu dengan perasaan negatif.

> `speechiness` adalah Mengukur seperapa jelas lirik dalam lagu


## DATA UNDERSTANDING

#### 1. Apakah ada missing value?
#### 2. Apakah ada duplikasi?
#### 3. Bagaimana distribusi popularity?
#### 4. Genre apa yang paling populer?
#### 5. Kolom numerik mana yang berkaitan dengan popularity?

In [ ]:
# ringkasan kolom numerik
df.describe()

In [ ]:
# ringkasan kolom kategorikal
df.describe(include='object')

In [ ]:
# Mencari nilai kosong
print("Missing Values:")
print(df.isna().sum())

In [ ]:
# Mencari duplikasi baris penuh
print("Duplicate Values:")
print(df.duplicated().sum())

In [ ]:
# jumlah nilai unik di setiap kolom
df.nunique()

## Data Cleaning

> Karena jumlah missing value sangat kecil dibanding total data, solusinya adalah menghapus baris yang memiliki missing value.

> Duplikasi baris penuh juga dihapus agar lagu yang sama tidak dihitung berulang dalam analisis popularity dan genre.

In [ ]:
df_clean = df.copy()

if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 0'])

print("Data awal:", df_clean.shape)

In [ ]:
# hapus missing value
df_clean = df_clean.dropna()

# hapus duplikasi baris penuh
df_clean = df_clean.drop_duplicates()

# reset index
df_clean = df_clean.reset_index(drop=True)

print("Data setelah cleaning:", df_clean.shape)

In [ ]:
print("Missing Values setelah cleaning:")
print(df_clean.isna().sum())

print("\nDuplicate Values setelah cleaning:")
print(df_clean.duplicated().sum())

## Feature Engineering

> `duration_ms` diubah menjadi `duration_min` agar lebih mudah dibaca.

> `popularity_category` dibuat untuk membagi lagu menjadi Low, Medium, dan High Popularity.

> `is_popular` dibuat sebagai target klasifikasi. Lagu dianggap populer jika `popularity >= 70`.

In [ ]:
df_clean['duration_min'] = df_clean['duration_ms'] / 60000

# -1 di pake karena biasanya popularity ada yang nilainya 0, kalau 0 0 nanti tidak di hitung
df_clean['popularity_category'] = pd.cut(
    df_clean['popularity'],
    bins=[-1, 39, 69, 100],
    labels=['Low', 'Medium', 'High']
)

df_clean['is_popular'] = (df_clean['popularity'] >= 70).astype(int)

df_clean[['track_name', 'artists', 'popularity', 'popularity_category', 'is_popular']].head()

# Target Variable: Popularity

In [ ]:
df_clean['popularity_category'].value_counts()

In [ ]:
df_clean['popularity_category'].value_counts(normalize=True) * 100

In [ ]:
print("Data popular atau tidak:")
print(df_clean['is_popular'].value_counts())

print("\nPersentase data:")
print(df_clean['is_popular'].value_counts(normalize=True) * 100)

> Pada project ini, target analisisnya adalah lagu populer.

> Lagu dengan `popularity >= 70` dianggap sebagai lagu populer karena berada pada skor popularitas tinggi.

> Setelah target dibuat, analisis selanjutnya akan mencari perbedaan karakteristik antara lagu populer dan tidak populer.

## Visualisasi Popularity

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df_clean, x='popularity_category', order=['Low', 'Medium', 'High'])
plt.title('Distribusi Kategori Popularity')
plt.xlabel('Popularity Category')
plt.ylabel('Jumlah Lagu')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean, x='popularity', bins=30, kde=True)
plt.title('Distribusi Popularity Score')
plt.xlabel('Popularity')
plt.ylabel('Jumlah Lagu')
plt.show()

In [ ]:
df_clean[df_clean['popularity'] == 0][['track_name', 'artists', 'popularity', 'track_genre']].head(20)

## Analisis Genre

In [ ]:
top_genre_popularity = (
    df_clean.groupby('track_genre')['popularity']
    .agg(['count', 'mean'])
    .sort_values('mean', ascending=False)
    .head(15)
)

top_genre_popularity

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_genre_popularity.reset_index(),
    y='track_genre',
    x='mean'
)
plt.title('Top 15 Genre Berdasarkan Rata-rata Popularity')
plt.xlabel('Rata-rata Popularity')
plt.ylabel('Genre')
plt.show()

## Explicit vs Non-Explicit

In [ ]:
# lagu dengan lyric kasar
explicit_summary = df_clean.groupby('explicit')['popularity'].agg(['count', 'mean', 'median'])
explicit_summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=df_clean, x='explicit', y='popularity')
plt.title('Rata-rata Popularity: Explicit vs Non-Explicit')
plt.xlabel('Explicit')
plt.ylabel('Popularity')
plt.show()

## Explicit vs Non-Explicit

> karena data tak seimbang, aku ingin melihat jika data seimbang apakah tetap sama hasilnya


In [ ]:
explicit_true = df_clean[df_clean['explicit'] == True]
explicit_false = df_clean[df_clean['explicit'] == False].sample(
    n=len(explicit_true),
    random_state=42
)

balanced_explicit = pd.concat([explicit_true, explicit_false])

In [ ]:
balanced_explicit.groupby('explicit')['popularity'].agg(['count', 'mean', 'median'])


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=balanced_explicit, x='explicit', y='popularity')
plt.title('Rata-rata Popularity: Explicit vs Non-Explicit')
plt.xlabel('Explicit')
plt.ylabel('Popularity')
plt.show()

> Ternyata setelah jumlah sampel diseimbangkan, lagu explicit memiliki rata-rata dan median popularity yang sedikit lebih tinggi dibanding lagu non-explicit dalam dataset ini.

> Ada indikasi lagu explicit cenderung memiliki popularity lebih tinggi, tetapi perlu analisis lanjutan untuk memastikan pengaruhnya.







In [ ]:
genre_compare = pd.crosstab(
    balanced_explicit['track_genre'],
    balanced_explicit['explicit']
)

genre_compare

## Mencari Kolom yang Berkaitan dengan Lagu Populer

In [ ]:
# kebanyakan kolom numerik
audio_cols = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_min'
]

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(audio_cols):
    sns.boxplot(data=df_clean, x='is_popular', y=col, ax=axes[i])
    axes[i].set_title(f'{col} vs Popular')
    axes[i].set_xlabel('is_popular')

plt.tight_layout()
plt.show()

In [ ]:
# heatmap korelasi
corr_cols = audio_cols + ['popularity', 'is_popular']

plt.figure(figsize=(12, 8))
sns.heatmap(
    df_clean[corr_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title('Correlation Heatmap Audio Features dan Popularity')
plt.show()

## Persiapan Data untuk Modeling

> Di bagian ini targetnya adalah `is_popular`.

> Kolom teks seperti `track_id`, `artists`, `album_name`, dan `track_name` tidak digunakan untuk modeling awal karena nilainya terlalu unik dan tujuan analisis ini adalah mencari karakteristik lagu mana yang populer, mecari nilai seperti (danceability, loudness, dll.)

> `track_genre` diubah menjadi angka menggunakan one hot encoding.

In [ ]:
model_cols = audio_cols + [
    'explicit', 'key', 'mode', 'time_signature',
    'track_genre', 'popularity', 'is_popular'
]

model_df = df_clean[model_cols].copy()

model_df['explicit'] = model_df['explicit'].astype(int)

model_df = pd.get_dummies(model_df, columns=['track_genre'], drop_first=True)

model_df.head()

### Encode Target

In [ ]:
X = model_df.drop(columns=['popularity', 'is_popular'])
y = model_df['is_popular']

feature_names = X.columns.tolist()

## Pemisahan Data untuk Training

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Pakai Model yang Paling Simple Dulu yaitu Logistic Regression

> Karena targetnya adalah populer atau tidak populer, maka bentuk problem-nya adalah classification.

> Logistic Regression dipakai sebagai baseline model karena mudah dijelaskan dan cocok untuk memulai analisis klasifikasi.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)
print(classification_report(y_test, y_pred_lr))

## Model RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

## Model XGB

In [ ]:
try:
    from xgboost import XGBClassifier

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model_xgb = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss'
    )

    model_xgb.fit(X_train, y_train)

    y_pred_xgb = model_xgb.predict(X_test)
    print(classification_report(y_test, y_pred_xgb))

except ImportError:
    print("XGBoost belum tersedia di environment ini. Lewati model XGB atau install xgboost terlebih dahulu.")

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

## Confusion Matrix

In [ ]:
models = {
    'Logistic Regression': {
        'model': model_lr,
        'X_test': X_test_scaled
    },
    'Random Forest': {
        'model': model_rf,
        'X_test': X_test
    },
    'XGBoost': {
        'model': model_xgb,
        'X_test': X_test
    }
}


In [ ]:
for name, item in models.items():
    model = item['model']
    X_eval = item['X_test']
    
    y_pred = model.predict(X_eval)
    
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Tidak Populer', 'Populer'],
        yticklabels=['Tidak Populer', 'Populer']
    )
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Prediksi')
    plt.ylabel('Aktual')
    plt.show()


## ROC_AUC & PR-AUC

In [ ]:
results = []

for name, item in models.items():
    model = item['model']
    X_eval = item['X_test']
    
    y_score = model.predict_proba(X_eval)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_score)
    pr_auc = average_precision_score(y_test, y_score)
    
    results.append({
        'model': name,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    })

auc_results = pd.DataFrame(results)
auc_results


In [ ]:
plt.figure(figsize=(8, 6))

for name, item in models.items():
    model = item['model']
    X_eval = item['X_test']
    
    y_score = model.predict_proba(X_eval)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = roc_auc_score(y_test, y_score)
    
    plt.plot(fpr, tpr, label=f'{name} AUC = {roc_auc:.3f}')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))

for name, item in models.items():
    model = item['model']
    X_eval = item['X_test']
    
    y_score = model.predict_proba(X_eval)[:, 1]
    
    precision, recall, _ = precision_recall_curve(y_test, y_score)
    pr_auc = average_precision_score(y_test, y_score)
    
    plt.plot(recall, precision, label=f'{name} PR-AUC = {pr_auc:.3f}')

plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.show()


## Hasil dari CM dan ROC-AUC, PRD-AUC

> Confusion matrix threshold 0.5:
LR paling banyak menangkap populer
XGB cukup seimbang
RF terlalu jarang menebak populer

> ROC-AUC dan PR-AUC:
RF punya kemampuan ranking/pemisahan terbaik

> Random Forest memiliki nilai ROC-AUC dan PR-AUC tertinggi, sehingga secara umum paling baik dalam membedakan lagu populer dan tidak populer. Namun pada threshold default 0.5, Random Forest terlalu konservatif dalam memprediksi lagu populer, sehingga recall kelas populer rendah. Oleh karena itu, Random Forest berpotensi menjadi model terbaik jika threshold prediksinya disesuaikan.








## Feature Importance

> Feature importance digunakan untuk melihat kolom mana yang paling berpengaruh terhadap prediksi lagu populer.

### Model Logistic Regression

In [ ]:
coef = pd.Series(
    np.abs(model_lr.coef_[0]),
    index=feature_names
)

coef.nlargest(10).sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Fitur Paling Berpengaruh (Logistic Regression)')
plt.xlabel('Absolute Coefficient')
plt.tight_layout()
plt.show()

### Model RandomForest

In [ ]:
feat_importance_rf = pd.Series(
    model_rf.feature_importances_,
    index=feature_names
)

feat_importance_rf.nlargest(10).sort_values().plot(kind='barh')
plt.title('Top 10 Fitur Paling Berpengaruh (RandomForest)')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

### Model XGB

In [ ]:
if 'model_xgb' in globals():
    feat_importance_xgb = pd.Series(
        model_xgb.feature_importances_,
        index=feature_names
    )

    feat_importance_xgb.nlargest(10).sort_values().plot(kind='barh')
    plt.title('Top 10 Fitur Paling Berpengaruh (XGB)')
    plt.xlabel('Feature Importance')
    plt.tight_layout()
    plt.show()
else:
    print("Model XGB tidak tersedia, feature importance XGB dilewati.")

# Kesimpulan

In [ ]:
total_tracks = len(df_clean)
popular_tracks = df_clean['is_popular'].sum()
popular_rate = df_clean['is_popular'].mean() * 100

top_genres = (
    df_clean.groupby('track_genre')['popularity']
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

explicit_mean = df_clean.groupby('explicit')['popularity'].mean()

print("RINGKASAN BISNIS")
print(f"Total lagu setelah cleaning: {total_tracks:,}")
print(f"Jumlah lagu populer: {popular_tracks:,}")
print(f"Persentase lagu populer: {popular_rate:.2f}%")

print("\nTop 5 genre berdasarkan rata-rata popularity:")
print(top_genres)

print("\nRata-rata popularity explicit vs non-explicit:")
print(explicit_mean)

## Jawaban Bisnis

1. **Bagaimana distribusi popularity lagu Spotify?**  
   Mayoritas lagu (57.2%) masuk kategori Low popularity (0–39), diikuti Medium (37.9%), dan hanya **4.8% lagu yang tergolong High popularity (≥70)**. Ini menunjukkan bahwa lagu populer adalah "barang langka" di Spotify.

2. **Genre apa yang memiliki rata-rata popularity tertinggi?**  
   Top 5 genre dengan popularity tertinggi adalah: **pop-film (59.3)**, **k-pop (57.0)**, **chill (53.7)**, **sad (52.4)**, dan **grunge (49.6)**. Genre pop-film unggul karena biasanya terhubung dengan film populer yang sudah punya audiens besar.

3. **Apakah lagu explicit lebih populer?**  
   Ya, lagu explicit memiliki rata-rata popularity **sedikit lebih tinggi** dibanding non-explicit. Namun perbedaannya tidak terlalu signifikan, jadi explicit bukan faktor utama popularity.

4. **Audio features apa yang membedakan lagu populer dan tidak populer?**  
   Berdasarkan correlation heatmap dan feature importance dari model:
   - Lagu populer cenderung memiliki **loudness lebih tinggi** (korelasi positif terkuat)
   - **Acousticness dan instrumentalness lebih rendah** (korelasi negatif)
   - **Danceability dan energy lebih tinggi**
   - Tempo dan valence tidak terlalu berpengaruh signifikan

5. **Apa rekomendasi playlist strategy?**  
   - **Playlist "Hits"**: prioritaskan lagu dengan loudness tinggi, danceability tinggi, acousticness rendah — profil audio yang terbukti berkorelasi dengan popularity.
   - **Playlist berdasarkan genre**: pop-film, k-pop, dan chill bisa jadi genre andalan untuk menarik pengguna.
   - **Discovery playlist untuk artis baru**: gunakan model prediksi popularity berdasarkan audio features untuk menemukan lagu potensial dari artis yang belum terkenal.
